Define your RAG corpus
Job postings (cleaned descriptions)
Optional: company “about” pages / values
Optional: strong resume bullet examples by role
Build one ingestion pipeline in your RAG notebook
Load docs
Chunk (start simple: 400–800 chars, small overlap)
Embed + store in a vector DB (FAISS or Chroma is fine for local)
Add one retrieval function
Input: job_title + job_description
Output: top-k relevant chunks (e.g., 4–8)
Inject retrieved context into your existing prompt flow
In service.py, retrieve context before build_resume_prompt / build_cover_letter_prompt
Pass retrieved text as an additional section in the prompt:
RETRIEVED CONTEXT: ...
Keep hard rule: “Do not invent candidate facts; candidate JSON is source of truth.”
Add evaluation in notebook
Compare without_rag vs with_rag on 5–10 jobs
Score: relevance, factuality, keyword alignment, hallucination rate

# Calling the application pipeline and LLM is producing the blank template set for candiates to enter resume information

In [35]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'HireMe.AI-V1':
    repo_root = cwd.parent
    project_dir = cwd
else:
    repo_root = cwd
    project_dir = repo_root / 'HireMe.AI-V1'

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from service import run_pipeline

out = run_pipeline(
    repo_root=repo_root,
    candidate_json_path=project_dir / 'candidate_profile.json',
    job_description='Paste full JD here',
    job_title='Data Analyst',
    company_name='Acme',
    output_dir=repo_root / 'notebooks' / 'outputs',
    use_tool_calling=False,
)

print(out['resume_md'][:600])
print(out['cover_letter_md'][:600])


# [Name]
[Email] | [Phone] | [Website]
[Date]

Hiring Manager  
Acme  
City, State

Dear Hiring Manager,

I am writing to express my interest in the Data Analyst position at Acme. I am drawn to Acme's commitment to using data to drive decisions and enhance product outcomes.

I bring a strong foundation in data analysis and collaboration, and I am eager to contribute to the data team's efforts at Acme. I am excited to learn more about the role's requirements and how I can support the team.

I am particularly interested in Acme because of its focus on delivering data-driven insights to improve business results. I’m excited by the opp


# Kaggle Dataset on Linkedin Job postings from 2023-2024

URL: https://www.kaggle.com/datasets/arshkon/linkedin-job-postings

In [36]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = 'postings.csv'
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'arshkon/linkedin-job-postings',
    file_path,
)

DEV_N_ROWS = 200
RANDOM_SEED = 42
df_dev = df.sample(min(len(df), DEV_N_ROWS), random_state=RANDOM_SEED).copy()

print('full rows:', len(df), '| dev rows:', len(df_dev))
display(df_dev.head(3))


/var/folders/5k/92wr5rmn2h7b9jndmkgm8rpc0000gn/T/ipykernel_55090/2913771257.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


full rows: 123849 | dev rows: 200


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
73989,3902944011,Current Power,Senior Automation Engineer - Power Systems,The Senior Automation / Power Systems Engineer...,NaN,NaN,"Houston, TX",760913.0,22.0,NaN,...,NaN,1.713280e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,77002.0,48201.0
59308,3901960222,DISH Network,DISH Installation Technician - Field,"Company Summary\n\nDISH, an EchoStar Company, ...",19.75,HOURLY,"Orange, TX",4296.0,5.0,NaN,...,NaN,1.713478e+12,jobs.dish.com,0,FULL_TIME,USD,BASE_SALARY,41080.0,77630.0,48361.0
44663,3900944095,"Coca-Cola Bottling Company UNITED, Inc.",Order Builder,Division: North Alabama\n\nDepartment : Oxford...,NaN,NaN,"Oxford, AL",136791.0,4.0,NaN,...,NaN,1.713389e+12,careers.cokeonena.com,0,FULL_TIME,NaN,NaN,NaN,36203.0,1015.0


# Document Construction

Converts each row into a Document with:
page content: title/company/location/description text
metadata: title/company/location

In [37]:
from langchain_core.documents import Document

def row_to_doc(row):
    title = str(row.get('title', ''))
    company = str(row.get('company_name', ''))
    location = str(row.get('location', ''))
    desc = str(row.get('description', ''))
    text = f"Job Title: {title}\nCompany: {company}\nLocation: {location}\nDescription:\n{desc}"
    return Document(
        page_content=text,
        metadata={'title': title, 'company': company, 'location': location}
    )

docs = [row_to_doc(r) for _, r in df_dev.iterrows() if str(r.get('description', '')).strip()]
print('docs:', len(docs))


docs: 200


# Chunking
splits documents into chunks

for sake of testing, using a smaller sample size to run 

In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

MAX_CHUNKS = 400
chunks = chunks[:MAX_CHUNKS]
print('chunks (capped):', len(chunks))


chunks (capped): 400


# Embedding + Vectorization

Embeds chunks with text-embedding-3-small from OPENAI

Vector stores using FAISS

Testing between ChromaDB

In [39]:
import time

t0 = time.time()
emb = OpenAIEmbeddings(model='text-embedding-3-small')
vs = FAISS.from_documents(chunks, emb)

print(f"Built vectorstore in {time.time()-t0:.1f}s")


Built vectorstore in 3.6s


# Query and Retrieval

Querying stored data for job information, this case data science roles

using top k=4

In [40]:
query = "Data Science roles that uses python and SQL."
hits = vs.similarity_search(query, k=4)

for i, h in enumerate(hits, 1):
    print(f"\n--- hit {i} ---")
    print(h.metadata)
    print(h.page_content[:350])


--- hit 1 ---
{'title': 'Sr. Advisory Data Engineer', 'company': 'Syniverse', 'location': 'United States'}
Bachelor’s degree in Computer Science, Statistics, Informatics, Information Systems, or another quantitative field Minimum of 12+ years of experience in a Data Engineer role or related fieldExperience using the following software/tools: Big data tools: Hadoop, Palantir, Spark, Kafka, etc. Relational SQL: Postgres, Oracle, etc. Data pipeline and wor

--- hit 2 ---
{'title': 'AI Engineer, Biologics Engineering, Oncology R&D', 'company': 'AstraZeneca', 'location': 'Gaithersburg, MD'}
Essential For The Role


Bachelor’s Degree or comparable experience with a minimum of 0+ years of experience in a Computer Science or Data Management related fieldTrack record of implementing software engineering best practices for multiple use cases.Experience of automation of the entire machine learning model lifecycle.Experience with optimizatio

--- hit 3 ---
{'title': 'Sr. Advisory Data Engineer', 

# Prompt Augmentation

Using testing candidate profile json builds normal cover letter/resume prompt

Appends Retrieved job market context + guardrail
Uses relevant context to build resume while keeping candidate profile as truth source 

In [41]:
import json
from dotenv import load_dotenv
from models import CandidateProfile, JobPosting
from prompts import build_resume_prompt, build_cover_letter_prompt
from agents import make_resume_writer_llm, make_cover_letter_writer_llm

load_dotenv(repo_root / '.env', override=True)

retrieved_context = '\n\n'.join(h.page_content for h in hits)

candidate_path = project_dir / 'candidate_profile.testing.json'
candidate_data = json.loads(candidate_path.read_text(encoding='utf-8'))
profile = CandidateProfile(**candidate_data)

job_data = {
    'job_title': hits[0].metadata.get('title', 'Data Analyst'),
    'company_name': hits[0].metadata.get('company', 'Unknown Company'),
    'job_description': hits[0].page_content,
}
job = JobPosting(**job_data)

resume_template = (project_dir / 'Templates' / 'resume_template.md').read_text(encoding='utf-8')
cover_template = (project_dir / 'Templates' / 'cover_letter_template.md').read_text(encoding='utf-8')

resume_prompt = build_resume_prompt(profile, job, resume_template)
cover_prompt = build_cover_letter_prompt(profile, job, cover_template)

rag_guardrail = (
    '\n\nRETRIEVED JOB MARKET CONTEXT:\n' + retrieved_context +
    '\n\nUse retrieved context only for role/company relevance and keyword alignment. '
    'Do NOT invent candidate facts. Candidate JSON is the only source for experience, dates, and metrics.'
)

resume_prompt_rag = resume_prompt + rag_guardrail
cover_prompt_rag = cover_prompt + rag_guardrail

resume_llm = make_resume_writer_llm()
cover_llm = make_cover_letter_writer_llm()

print('Using candidate:', candidate_path.name)
print('Target:', job.job_title, '@', job.company_name)
print('Retrieved context chars:', len(retrieved_context))


Using candidate: candidate_profile.testing.json
Target: Sr. Advisory Data Engineer @ Syniverse
Retrieved context chars: 3588


In [42]:
resume_md_rag = resume_llm.invoke(resume_prompt_rag).content.strip()
cover_md_rag = cover_llm.invoke(cover_prompt_rag).content.strip()

print('RAG resume preview:\n')
print(resume_md_rag[:1200])
print('\n' + '=' * 80 + '\n')
print('RAG cover letter preview:\n')
print(cover_md_rag[:1200])


RAG resume preview:

# Jordan Lee
jordan.lee@example.com | 704-555-0189 | https://www.linkedin.com/in/jordanlee-data

## Summary
Data analyst with 4+ years building KPI dashboards, automating reporting workflows, and translating business questions into actionable insights. Strong expertise in SQL, Python, and BI tools with a proven track record of improving decision speed and data quality. Eager to leverage data engineering fundamentals and cross-functional collaboration in a Sr. Advisory Data Engineer role at Syniverse.

## Work Experience
### Senior Data Analyst - BlueArc Logistics
June 2022 - Present
- Built and maintained Tableau dashboards for operations and finance, reducing weekly reporting time by 35%.
- Developed SQL data models for shipment performance tracking across 12 regions.
- Automated recurring KPI exports with Python scripts, saving 10+ analyst hours per week.
- Partnered with leadership to define SLA metrics and identify root causes of delivery delays.

### Data Anal

In [43]:
out_dir = repo_root / 'notebooks' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

resume_out = out_dir / 'resume_output_rag.md'
cover_out = out_dir / 'cover_letter_output_rag.md'
resume_out.write_text(resume_md_rag, encoding='utf-8')
cover_out.write_text(cover_md_rag, encoding='utf-8')

print('Saved:', resume_out)
print('Saved:', cover_out)


Saved: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/notebooks/outputs/resume_output_rag.md
Saved: /Users/jefferysengsy/Documents/GitHub/Spring-2026-DSBA-6010-Group-13-HireMe-AI/notebooks/outputs/cover_letter_output_rag.md
